# Apigee Template: REST-AI-GenerateContent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gcp-samples/apigee-template-repository/blob/main/notebooks/REST-AI-GenerateContent.ipynb)

**Template Name:** `REST-AI-GenerateContent`  
**Description:** Google Cloud Vertex AI & Gemini `generateContent` API Gateway proxy supporting multimodal inputs, streaming responses, intelligent model routing, model checks, failover, and token usage analytics.

### Key Capabilities:
- **Model Pre-Processing & Checks:** Apigee inspects the incoming request (`JS-CheckModel`), verifies requested models, and performs model mapping/aliasing (e.g., mapping `gemini-flash-latest` to `gemini-3.8-flash`).
- **Token Usage Analytics & Data Collection:** On response delivery, Apigee extracts token consumption metrics from `usageMetadata` and records them directly into Apigee Data Collectors (`dc_ai_prompt_token_count`, `dc_ai_response_token_count`, `dc_ai_total_token_count`, `dc_ai_model`). These metrics feed Apigee Custom Reports for real-time cost, latency, and consumption tracking.
- **Automated Google IAM Token Minting:** If requests arrive without an Authorization header, Apigee's `AM-SetGoogleToken` automatically mints a Google Cloud OAuth token via the Apigee service account (`roles/aiplatform.user`), eliminating client-side credential sharing.
- **Streaming & Failover Support:** Transparently proxies standard unary `generateContent` and streaming `streamGenerateContent` responses with failover protection.

### Documentation & References:
- [Google Cloud Vertex AI generateContent API Documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/inference)
- [Google Gemini API generateContent Documentation](https://ai.google.dev/api/generate-content)
- [Apigee Feature Templater (aft) GitHub](https://github.com/apigee/apigee-templater)
- [Apigee Data Collectors Documentation](https://cloud.google.com/apigee/docs/api-platform/analytics/data-collectors)
- [Apigee Custom Reports Documentation](https://cloud.google.com/apigee/docs/api-platform/analytics/reports-overview)

### Workflow:
1. **Configuration & Authentication:** Enter your `GOOGLE_CLOUD_PROJECT`, `APIGEE_ENV`, and `GEMINI_API_KEY`.
2. **Setup, Initialize Resources & Deploy:** Install `aft`, run `sh/initialize.sh` (service account, IAM roles, data collectors, reports), and deploy `REST-AI-GenerateContent.yaml`.
3. **Test Initialized GenerateContent API:** Send unary generateContent, alias mapping/model check, and streaming requests using `APIGEE_HOST`.

---

## 1. Configuration & Authentication

Specify your Google Cloud Project ID (Apigee Organization), Apigee environment, and your Gemini API Key.

In [ ]:
# @title 1. Configuration & Authentication
import os

# @markdown Enter your Google Cloud Project ID (Apigee Organization):
GOOGLE_CLOUD_PROJECT = "your_apigee_org"  # @param {type:"string"}
APIGEE_ENV = "dev"  # @param {type:"string"}
GEMINI_API_KEY = "your_gemini_api_key"  # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
os.environ["APIGEE_ORG"] = GOOGLE_CLOUD_PROJECT
os.environ["APIGEE_ENV"] = APIGEE_ENV
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["APIGEE_SA"] = f"apigee-service@{GOOGLE_CLOUD_PROJECT}.iam.gserviceaccount.com"

try:
    from google.colab import auth
    auth.authenticate_user()
    print(f"Authenticated with Google Cloud for project: {GOOGLE_CLOUD_PROJECT}")
except ImportError:
    print("Running outside Google Colab.")


## 2. Setup Tools, Initialize Resources & Deploy Template

Downloads `templates/REST-AI-GenerateContent.yaml` and `sh/initialize.sh` (if running standalone in Colab), installs `aft`, runs resource initialization (service accounts, IAM bindings, data collectors, custom reports), sets `APIGEE_HOST`, and deploys the template.

In [ ]:
# @title 2. Setup, Initialize & Deploy Template
import os
import subprocess

# 1. Install Apigee Feature Templater (aft) CLI if needed
!which aft >/dev/null 2>&1 || curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh

# 2. Download template and initialize script if running standalone
REPO_RAW = "https://raw.githubusercontent.com/gcp-samples/apigee-template-repository/main"
TEMPLATE_FILE = "REST-AI-GenerateContent.yaml" if os.path.exists("REST-AI-GenerateContent.yaml") else "templates/REST-AI-GenerateContent.yaml" if os.path.exists("templates/REST-AI-GenerateContent.yaml") else "REST-AI-GenerateContent.yaml"
os.environ["TEMPLATE_FILE"] = TEMPLATE_FILE

if not os.path.exists(TEMPLATE_FILE):
    !curl -fsSL -O {REPO_RAW}/templates/REST-AI-GenerateContent.yaml

if not os.path.exists("sh/initialize.sh"):
    !mkdir -p sh && curl -fsSL -o sh/initialize.sh {REPO_RAW}/sh/initialize.sh

# 3. Run initialization script (sets up service account, IAM bindings, data collectors, reports)
!bash sh/initialize.sh

# 4. Resolve APIGEE_HOST using aft describe
cmd = 'aft describe --project "$GOOGLE_CLOUD_PROJECT" -f json | jq --raw-output ".environmentGroups[] | select(any(.attachments[]; .environment == \"$APIGEE_ENV\")) | .hostnames[0]"'
try:
    host = subprocess.check_output(cmd, shell=True, text=True).strip()
    if host and host != "null":
        os.environ["APIGEE_HOST"] = host
        print(f"APIGEE_HOST resolved to: {host}")
except Exception as e:
    print(f"Could not automatically resolve APIGEE_HOST via aft: {e}")

# 5. Deploy template with aft
!aft "$TEMPLATE_FILE" \
  --project="$GOOGLE_CLOUD_PROJECT" \
  --env="$APIGEE_ENV" \
  --sa="$APIGEE_SA"


## 3. Test Initialized API via APIGEE_HOST

Send requests to `https://${APIGEE_HOST}/v1/projects/${PROJECT_ID}/locations/global/publishers/google/models/{model}:{method}`.

- **Model Checks:** The proxy validates the requested model and applies model alias mapping (`gemini-flash-latest` -> `gemini-3.8-flash`).
- **Token Usage Recording:** Apigee captures prompt and candidate token counts from `usageMetadata` into Data Collectors (`dc_ai_prompt_token_count`, `dc_ai_response_token_count`, `dc_ai_total_token_count`).
- **Google Cloud IAM Minting:** Apigee's `AM-SetGoogleToken` policy attaches service account credentials to backend Vertex AI calls, or forwards caller-supplied authorization tokens and API keys.

In [ ]:
# @title Setup Test Client & APIGEE_HOST
import os
import json
import requests
import subprocess

PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "")
APIGEE_ENV = os.getenv("APIGEE_ENV", "dev")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

# Determine APIGEE_HOST
APIGEE_HOST = os.getenv("APIGEE_HOST")
if not APIGEE_HOST or APIGEE_HOST == "null":
    APIGEE_HOST = f"{PROJECT_ID}-{APIGEE_ENV}.apigee.net"
    os.environ["APIGEE_HOST"] = APIGEE_HOST

# Retrieve GCP access token if available for caller authorization
try:
    gcp_token = subprocess.check_output(["gcloud", "auth", "application-default", "print-access-token"], text=True).strip()
except Exception:
    try:
        gcp_token = subprocess.check_output(["gcloud", "auth", "print-access-token"], text=True).strip()
    except Exception:
        gcp_token = ""

print(f"APIGEE_HOST: {APIGEE_HOST}")
print(f"Base Project Path: https://{APIGEE_HOST}/v1/projects/{PROJECT_ID}")

def send_generate_content(model: str, prompt_text: str, method: str = "generateContent", stream: bool = False):
    """
    Sends a generateContent request through the Apigee Gateway to Google Cloud Vertex AI / Gemini.
    Apigee applies model checks/mapping, automatically attaches IAM credentials if needed,
    and logs token metrics into Apigee Data Collectors.
    """
    url = f"https://{APIGEE_HOST}/v1/projects/{PROJECT_ID}/locations/global/publishers/google/models/{model}:{method}"
    
    headers = {"Content-Type": "application/json"}
    if gcp_token:
        headers["Authorization"] = f"Bearer {gcp_token}"
    elif GEMINI_API_KEY:
        headers["x-goog-api-key"] = GEMINI_API_KEY

    payload = {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {"text": prompt_text}
                ]
            }
        ]
    }

    print(f"\n---> Sending [{model}:{method}] request to {url}...")
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30, stream=stream)
        print(f"HTTP Status: {response.status_code}")
        
        if stream:
            print("Streaming Response Chunks:")
            for chunk in response.iter_lines():
                if chunk:
                    print(chunk.decode("utf-8"))
            return None
        
        try:
            data = response.json()
            print(json.dumps(data, indent=2))
            
            # Display token usage recorded by Apigee
            usage = data.get("usageMetadata", {})
            if usage:
                print("\n[Token Usage Recorded by Apigee Data Collectors]")
                print(f"  Prompt Tokens:     {usage.get('promptTokenCount', 'N/A')}")
                print(f"  Candidates Tokens: {usage.get('candidatesTokenCount', 'N/A')}")
                print(f"  Total Tokens:      {usage.get('totalTokenCount', 'N/A')}")
            return data
        except Exception:
            print(response.text)
            return None
    except Exception as e:
        print("Request error:", e)
        return None


In [ ]:
# @title Test 1: Standard Content Generation (gemini-2.5-flash)
# Apigee verifies model parameters, routes to Vertex AI, and captures token analytics.
resp1 = send_generate_content(
    model="gemini-2.5-flash",
    prompt_text="Explain the concept of API governance in enterprise architectures in two concise sentences."
)


In [ ]:
# @title Test 2: Model Checks & Alias Mapping (gemini-flash-latest)
# The proxy checks the model parameter and evaluates ModelMapping (gemini-flash-latest -> gemini-3.8-flash).
resp2 = send_generate_content(
    model="gemini-flash-latest",
    prompt_text="What are 3 critical metrics to track when operating production AI models?"
)


In [ ]:
# @title Test 3: Streaming Content Generation (streamGenerateContent)
# Tests streamGenerateContent; Apigee handles SSE event flows and accumulates token usage.
resp3 = send_generate_content(
    model="gemini-2.5-flash",
    prompt_text="List 3 main benefits of using an API gateway for AI models.",
    method="streamGenerateContent",
    stream=True
)


## 4. Verify Apigee Analytics & Reports

Because `sh/initialize.sh` provisioned Data Collectors and Custom Reports, every request processed by `REST-AI-GenerateContent` records token analytics:

1. Open the [Google Cloud Apigee Console](https://console.cloud.google.com/apigee).
2. Navigate to **Analytics > Custom Reports**.
3. Inspect the pre-configured reports:
   - **`ai_token_cost_by_model_user`**: Aggregated AI token cost by model and user/developer.
   - **`ai_model_usage_latency`**: Real-time latency and call volume per model.
   - **`ai_token_counts_by_model_user`**: Prompt, response, and total token count breakdowns.
